# IAE-LSTM-KL: Gamma-only Anomaly-Detection PSD

**Idea:** train an autoencoder on **gamma-ray pulses only**. A neutron pulse has a bigger slow-decay tail, so it should look like
an *anomaly* to a model that has only ever seen gammas. The anomaly score is scored with the dataset paper's **figure of merit**
`FOM = S / (FWHM_gamma + FWHM_neutron)` and, more robustly, with **AUROC**.

Paper's classical baselines (raw dataset, no noise): ZC 1.01, FEPS 0.80, FGA 0.95, CC 1.33, LG 1.48, PCNN 1.66, **HQC-SCM 1.79**.

The model is **IAE-LSTM-KL** (Huang, Zhang, Zhang, Gao & Wan, arXiv:2404.19247): a convolutional autoencoder with an **LSTM gate module**
at the bottleneck, trained with a **Deep SVDD** loss plus a **KL** penalty. Training never sees a neutron pulse or a label.

**What changed from the first trimmed version (and why)** -- details and numbers are in the notes at the bottom:

1. **Pulses are normalised** (divided by their own peak, then by one global constant). In raw ADC units the LSTM gates saturated within ~20 optimiser steps and the latent became constant.
2. **The network uses stabilisers borrowed from the authors' released code** (BatchNorm with `affine=False`, bias-free convs, LeakyReLU; the BatchNorm on the latent comes from their plain autoencoder -- their LSTM model leaves it out). Without them the model only trains inside a narrow window of input scales and collapses on some seeds.
3. **Collapse can no longer pass silently.** The degenerate-FOM check also catches identical scores and empty mixture components, and a label-free health check runs after each training stage.
4. **Evaluation is on held-out gamma pulses** plus all neutrons, and reports AUROC for both the SVDD score and the reconstruction error.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

N_SAMPLES = 300   # samples per pulse (synthetic data only; overwritten when real data is loaded)

## 1. Data

Two options, controlled by one flag:

- **Real data** (`USE_REAL_TAMU_DATA = True`): the Texas A&M TOF-cut dataset from `github.com/NeutronNeutrinoSensing/PSDwithML` (3-clause BSD).
  Each CSV has 500 columns: 496 waveform samples at 2 ns, then event index, pulse height, tail-to-total ratio, and a class label
  (0 = gamma, 1 = neutron). The labels come from a time-of-flight cut, so the gamma label is independent of pulse shape.
- **Synthetic pulses** (`False`): a simple physics-motivated stand-in so the pipeline runs with no download.

The loader prints a sanity check -- the gamma file's labels should all be `0` and the neutron file's all `1`.
`y` (the true label) is **never used in training** -- only to score the result at the end.

In [ ]:
USE_REAL_TAMU_DATA = True   # False -> synthetic pulses

# ---- synthetic data ---------------------------------------------------------------------------
def simulate_pulse(n_samples, is_neutron, rng):
    # Fast rise, a fast decay shared by both particle types, plus a slow decay component
    # that is bigger for neutrons -- the physical difference PSD relies on.
    t0 = 50 + rng.integers(-5, 6)          # small trigger-time jitter
    t = np.arange(n_samples)
    tau_rise = rng.uniform(2, 4)
    tau_fast = rng.uniform(8, 12)
    tau_slow = rng.uniform(40, 60)
    amp = rng.uniform(0.6, 1.0)
    slow_frac = rng.uniform(0.25, 0.45) if is_neutron else rng.uniform(0.06, 0.15)
    fast_frac = 1 - slow_frac
    shape = np.where(
        t >= t0,
        (1 - np.exp(-(t - t0) / tau_rise)) *
        (fast_frac * np.exp(-(t - t0) / tau_fast) + slow_frac * np.exp(-(t - t0) / tau_slow)),
        0.0,
    )
    pulse = amp * shape + rng.normal(0, 0.01, n_samples)
    return pulse.astype(np.float32)

def make_synthetic_dataset(n_gamma, n_neutron, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    g = np.stack([simulate_pulse(n_samples, False, rng) for _ in range(n_gamma)])
    n = np.stack([simulate_pulse(n_samples, True, rng) for _ in range(n_neutron)])
    X = np.concatenate([g, n]).astype(np.float32)
    y = np.concatenate([np.zeros(n_gamma), np.ones(n_neutron)]).astype(np.int64)  # 0=gamma, 1=neutron
    idx = rng.permutation(len(X))
    return X[idx], y[idx]

# ---- real data (TAMU CoMPASS TOF-cut CSVs) -------------------------------------------------------
def find_file(keyword, files):
    matches = [f for f in files if keyword in f.lower()]
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f"Expected exactly one file containing '{keyword}' among {files} -- "
                     f"found {len(matches)}. Set the path manually if auto-detection failed.")

def load_tamu_csv(path):
    # 500 columns: 496 waveform samples, then event index, pulse height, tail-to-total ratio, label.
    df = pd.read_csv(path, header=None, low_memory=False)
    try:
        df.iloc[0].astype(float)
    except (ValueError, TypeError):
        df = df.iloc[1:].reset_index(drop=True)  # first row was a text header -- drop it
    assert df.shape[1] == 500, f"Expected 500 columns, got {df.shape[1]} in {path} -- check the file."
    df = df.astype(float)
    waveforms = df.iloc[:, :496].to_numpy(dtype=np.float32)
    label = df.iloc[:, -1].to_numpy(dtype=np.int64)
    return waveforms, label

In [ ]:
if USE_REAL_TAMU_DATA:
    import os, glob, subprocess

    if not os.path.isdir("PSDwithML"):
        result = subprocess.run(
            ["git", "clone", "https://github.com/NeutronNeutrinoSensing/PSDwithML.git"],
            capture_output=True, text=True,
        )
        print(result.stdout, result.stderr)

    data_files = sorted(glob.glob("PSDwithML/data/*.csv"))
    print("CSV files found in data/:", data_files)

    gamma_path = find_file("gamma", data_files)
    neutron_path = find_file("neutron", data_files)
    print("Using gamma file:  ", gamma_path)
    print("Using neutron file:", neutron_path)

    gamma_wave, gamma_label = load_tamu_csv(gamma_path)
    neutron_wave, neutron_label = load_tamu_csv(neutron_path)

    print(f"\ngamma file:   {gamma_wave.shape[0]} pulses, {gamma_wave.shape[1]} samples, "
          f"labels present: {np.unique(gamma_label)}")
    print(f"neutron file: {neutron_wave.shape[0]} pulses, {neutron_wave.shape[1]} samples, "
          f"labels present: {np.unique(neutron_label)}")
    if not (np.all(gamma_label == 0) and np.all(neutron_label == 1)):
        print("\nWARNING: labels don't match the expected pattern (gamma file should be all 0s, "
              "neutron file all 1s). Check the column-order assumption in load_tamu_csv, or that "
              "gamma_path/neutron_path weren't swapped.")

    X = np.concatenate([gamma_wave, neutron_wave]).astype(np.float32)
    y = np.concatenate([np.zeros(len(gamma_wave)), np.ones(len(neutron_wave))]).astype(np.int64)
    idx = np.random.default_rng(0).permutation(len(X))
    X, y = X[idx], y[idx]
    N_SAMPLES = X.shape[1]
else:
    X, y = make_synthetic_dataset(n_gamma=2500, n_neutron=800, n_samples=N_SAMPLES)

print(f"\nX.shape={X.shape}, N_SAMPLES={N_SAMPLES}, gamma={int((y==0).sum())}, neutron={int((y==1).sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rng_plot = np.random.default_rng(1)
for i in rng_plot.choice(np.where(y == 0)[0], size=min(15, (y == 0).sum()), replace=False):
    ax.plot(X[i], color="tab:blue", alpha=0.3)
for i in rng_plot.choice(np.where(y == 1)[0], size=min(15, (y == 1).sum()), replace=False):
    ax.plot(X[i], color="tab:red", alpha=0.3)
ax.plot([], [], color="tab:blue", label="gamma")
ax.plot([], [], color="tab:red", label="neutron")
ax.set_xlabel("sample"); ax.set_ylabel("amplitude (a.u.)"); ax.legend()
ax.set_title("Example raw pulses -- " + ("real TAMU data" if USE_REAL_TAMU_DATA else "synthetic"))
plt.show()

## 2. Preprocess: baseline correction + peak alignment

Digitized pulses sit on a small DC offset and don't all start at the same sample (trigger jitter). We subtract the baseline
(mean of the first 20 samples) and shift every pulse so its peak lands on the same sample.

In [ ]:
def align_and_baseline(X):
    baseline = X[:, :20].mean(axis=1, keepdims=True)
    X = X - baseline
    peak_idx = X.argmax(axis=1)
    target = int(np.median(peak_idx))
    aligned = np.zeros_like(X)
    for i in range(len(X)):
        aligned[i] = np.roll(X[i], target - peak_idx[i])
    return aligned, target

Xa, peak_ref = align_and_baseline(X)
print("peak reference sample:", peak_ref)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
example_idx = np.random.default_rng(0).choice(len(X), size=min(10, len(X)), replace=False)
for i in example_idx:
    axes[0].plot(X[i], alpha=0.5)
    axes[1].plot(Xa[i], alpha=0.5)
axes[0].set_title("Before: raw pulses\n(baseline offset + trigger jitter)")
axes[1].set_title("After: baseline-corrected + peak-aligned")
for ax in axes:
    ax.set_xlabel("sample"); ax.set_ylabel("amplitude (a.u.)")
axes[1].axvline(peak_ref, color="k", linestyle="--", alpha=0.5, label=f"peak_ref = {peak_ref}")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Gamma pool, held-out split, and normalisation

- **Real data:** the TOF-cut labels give a genuine, shape-independent gamma label, so the pool is every `y == 0` pulse.
- **Synthetic data:** no such label, so we use the charge-comparison (CC) factor -- tail charge over total charge -- and keep the 40% of pulses with the *lowest* values.

**Held-out split.** 70% of the pool is used for training and for the SVDD center. The other 30% of gammas plus every pulse outside the pool
(all neutrons) form the **evaluation set**, so the score is never measured on pulses the model trained on.

**Normalisation** (label-free). Each pulse is divided by its own peak, then everything is divided by one global constant so the training pulses have unit standard deviation.
Two reasons, both measured: the network only trains inside a window of input scales (raw ADC units saturate the gates; peak-normalised pulses at their natural std of ~0.09 leave the
reconstruction term too small and the SVDD term wins), and without removing amplitude the 2-D latent spends itself on amplitude (correlation ~ -0.99 with peak height), which says nothing about particle type.

In [ ]:
def cc_factor(X, peak_ref, tail_offset=15, start_offset=-5):
    total = X[:, peak_ref + start_offset:].sum(axis=1)
    tail = X[:, peak_ref + tail_offset:].sum(axis=1)
    return tail / (total + 1e-8)

if USE_REAL_TAMU_DATA:
    gamma_pool_mask = (y == 0)
    print("Using real TOF-based labels for the gamma pool:")
    print("  gamma-pool size:", gamma_pool_mask.sum(), "/", len(Xa))
else:
    cc = cc_factor(Xa, peak_ref)
    gamma_pool_mask = cc < np.percentile(cc, 40)
    print("gamma-pool size:", gamma_pool_mask.sum(), "/", len(Xa))
    print("(sanity check using the synthetic ground truth) pool purity:", (y[gamma_pool_mask] == 0).mean())

# 70/30 split of the pool: train_mask is all the model ever sees; eval_mask is everything else.
pool_idx = np.where(gamma_pool_mask)[0]
train_idx = np.random.default_rng(123).choice(pool_idx, size=int(0.7 * len(pool_idx)), replace=False)
train_mask = np.zeros(len(Xa), dtype=bool)
train_mask[train_idx] = True
eval_mask = ~train_mask
print(f"train pulses: {train_mask.sum()}   evaluation pulses: {eval_mask.sum()} "
      f"({int((y[eval_mask]==0).sum())} gamma, {int((y[eval_mask]==1).sum())} neutron)")

def normalise_pulses(Xa, fit_mask):
    # label-free: (1) divide each pulse by its own peak, (2) divide by one global constant -> unit std on fit_mask.
    shape = Xa / np.maximum(Xa.max(axis=1, keepdims=True), 1e-6)
    scale = float(shape[fit_mask].std())
    return (shape / scale).astype(np.float32), scale

Xn, PULSE_SCALE = normalise_pulses(Xa, train_mask)
print(f"normalised pulses: global constant = {PULSE_SCALE:.4f}, training-set std = {Xn[train_mask].std():.3f}")

## 4. The IAE-LSTM-KL model

A 1D convolutional encoder squeezes each pulse to a short **latent vector** `z`, and a mirrored decoder rebuilds the pulse from the LSTM-filtered vector `ẑ`.

**LSTM module (at the bottleneck).** The latent vector is treated as a time series with one step and zero previous state, which collapses the LSTM cell to a gated filter:
input gate `s = sigmoid(W_s·z + b_s)`, output gate `o = sigmoid(W_o·z + b_o)`, candidate `c̃ = tanh(W_c·z + b_c)`, cell `c = s ⊙ c̃`, output `ẑ = o ⊙ tanh(c)`.
Note `ẑ` is bounded: each coordinate lies in (-0.76, 0.76).

**Deep SVDD.** `L_svdd = mean ‖ẑ − c‖²`, and the same distance is the anomaly score at test time. `c` is not learned -- it's the mean latent vector over the training pulses,
computed once after a reconstruction-only pretraining stage (with a small `eps` nudge so no coordinate sits at exactly zero).

**KL term.** The paper doesn't spell out the per-vector formula; this uses `KL = 0.5·‖ẑ‖²`. Be aware this is an L2 penalty toward the origin: it is smallest for a collapsed ball and
**cannot by itself prevent hypersphere collapse**, and in the debugging runs it didn't (see the notes at the bottom). The authors' released code implements its "KL" term yet another way (`nn.KLDivLoss` against a random softmax target).
Treat `λ1` as inert until shown otherwise.

**Stabilisers** (`stabilise=True`, borrowed from the authors' released code): bias-free convs and latent projection, `BatchNorm(affine=False, eps=1e-4)` after every conv and on the latent, LeakyReLU(0.2). The latent BatchNorm comes from their plain autoencoder; their LSTM model leaves it out.
They make training insensitive to the input scale. `stabilise=False` is the original plain architecture.

**Total loss:** `L_total = L_svdd + λ1·L_kl + λ2·L_rec`.

In [ ]:
class LSTMModule(nn.Module):
    # Single-step LSTM cell used as a gated filter (h_{t-1} = c_{t-1} = 0, per the paper).
    def __init__(self, dim):
        super().__init__()
        self.input_gate = nn.Linear(dim, dim)    # -> s
        self.output_gate = nn.Linear(dim, dim)   # -> o
        self.candidate = nn.Linear(dim, dim)     # -> c_tilde

    def forward(self, z):
        s = torch.sigmoid(self.input_gate(z))
        o = torch.sigmoid(self.output_gate(z))
        c_tilde = torch.tanh(self.candidate(z))
        c = s * c_tilde
        return o * torch.tanh(c)


def compute_svdd_center(model, X_pool, device="cpu", eps=0.1):
    # Hypersphere center: mean latent vector over the training pulses. Coordinates too close to zero
    # are pushed out to +/- eps (standard Deep SVDD trick against trivial all-zeros solutions).
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_pool, dtype=torch.float32).unsqueeze(1).to(device)
        c = model.encode(X_t).mean(dim=0)
    sign = torch.where(c >= 0, torch.ones_like(c), -torch.ones_like(c))
    return torch.where(c.abs() < eps, sign * eps, c)


def svdd_score(z_hat, center):
    # SCORE = ||z_hat - c||^2 -- both the per-sample SVDD loss and the test-time anomaly score.
    return ((z_hat - center) ** 2).sum(dim=1)


def kl_to_standard_normal(z_hat):
    # KL( N(z_hat, I) || N(0, I) ) = 0.5 * ||z_hat||^2   (see the note above about what this does and doesn't do)
    return 0.5 * (z_hat ** 2).sum(dim=1)

In [ ]:
class IAE_LSTM_KL(nn.Module):
    # Conv encoder -> latent -> LSTM gate module -> decoder.
    # forward() returns (reconstruction, z_hat) since the SVDD/KL losses need the latent vector.
    def __init__(self, n_samples, latent_dim=8, stabilise=True):
        super().__init__()
        self.stabilise = stabilise
        bias = not stabilise
        norm1d = (lambda c: nn.BatchNorm1d(c, eps=1e-4, affine=False)) if stabilise else (lambda c: nn.Identity())
        act = (lambda: nn.LeakyReLU(0.2)) if stabilise else (lambda: nn.ReLU())
        def down(i, o): return [nn.Conv1d(i, o, kernel_size=7, stride=2, padding=3, bias=bias), norm1d(o), act()]
        def up(i, o):   return [nn.ConvTranspose1d(i, o, kernel_size=7, stride=2, padding=3, output_padding=1), norm1d(o), act()]

        self.encoder = nn.Sequential(*down(1, 16), *down(16, 32), *down(32, 64))
        self.eval()                                   # shape probe must not touch BatchNorm statistics
        with torch.no_grad():
            enc_out = self.encoder(torch.zeros(1, 1, n_samples))
        self.train()
        self._enc_shape = enc_out.shape[1:]           # (channels, length) without the batch dim
        flat_size = enc_out.numel()

        self.to_latent = nn.Linear(flat_size, latent_dim, bias=bias)
        self.latent_norm = norm1d(latent_dim)
        self.from_latent = nn.Linear(latent_dim, flat_size)
        self.lstm = LSTMModule(latent_dim)

        self.decoder = nn.Sequential(
            *up(64, 32), *up(32, 16),
            nn.ConvTranspose1d(16, 1, kernel_size=7, stride=2, padding=3, output_padding=1),
        )

    def encode(self, x):
        # z_hat: the LSTM-filtered latent vector used by both the decoder and the SVDD/KL losses.
        z = self.latent_norm(self.to_latent(self.encoder(x).flatten(1)))
        return self.lstm(z)

    def forward(self, x):
        input_len = x.shape[-1]
        z_hat = self.encode(x)
        z = self.from_latent(z_hat).view(-1, *self._enc_shape)
        out = self.decoder(z)
        if out.shape[-1] != input_len:               # conv/deconv arithmetic can be off by a few samples
            out = F.interpolate(out, size=input_len, mode="linear", align_corners=False)
        return out, z_hat


# quick shape check
_m = IAE_LSTM_KL(N_SAMPLES, latent_dim=8).to(device)
_x_hat, _z_hat = _m(torch.randn(6, 1, N_SAMPLES).to(device))
print("reconstruction:", tuple(_x_hat.shape), " latent (z_hat):", tuple(_z_hat.shape))
del _m, _x_hat, _z_hat

## 5. Training (training pulses only)

Two stages, as in the paper:

1. **Pretrain** -- reconstruction loss only, so the encoder has learned something before the SVDD center is computed.
2. **Joint training** -- the full loss. `weight_decay` is the paper's λ3 L2 term, passed straight to the optimiser.

After each stage a **label-free health check** compares the reconstruction error with the error of simply outputting the *average pulse* for every input.
A ratio near 1.0 means the decoder is ignoring the latent; a latent spread near 0 means the encoder has collapsed to one point. Both were the signature of the failed runs.

In [ ]:
def check_health(model, X_pool, device="cpu", stage=""):
    # Label-free. recon_ratio ~ 1 -> decoder ignores the latent; spread ~ 0 -> latent is constant across pulses.
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(X_pool, dtype=torch.float32).unsqueeze(1).to(device)
        x_hat, z_hat = model(Xt)
        rec = F.mse_loss(x_hat, Xt).item()
        spread = float(z_hat.var(dim=0).sum().sqrt())
    model.train()
    baseline = float(((X_pool - X_pool.mean(axis=0)) ** 2).mean())   # MSE of outputting the average pulse
    ratio = rec / max(baseline, 1e-12)
    problems = []
    if spread < 1e-3:
        problems.append("latent is (nearly) constant across pulses -- the encoder has collapsed")
    if ratio > 0.9:
        problems.append(f"reconstruction error is {ratio:.2f}x the 'average pulse' baseline -- the decoder is ignoring the latent")
    print(f"  [{stage}] recon/baseline = {ratio:.3f}   latent spread = {spread:.4f}   "
          + ("WARNING: " + "; ".join(problems) if problems else "ok"))
    return ratio, spread


def train_iae_lstm_kl(model, X_train, n_pretrain_epochs=15, n_train_epochs=15,
                       batch_size=64, lr=1e-3, weight_decay=1e-5,
                       lambda1=0.01, lambda2=1.0, device="cpu", verbose=True):
    X_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)    # (N, 1, L)
    # BatchNorm needs more than one sample per batch, so a trailing 1-sample batch is dropped when stabilised.
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_tensor),
                                          batch_size=batch_size, shuffle=True, drop_last=model.stabilise)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Stage 1: reconstruction-only pretraining
    model.train()
    for epoch in range(n_pretrain_epochs):
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(batch)
            F.mse_loss(x_hat, batch).backward()
            optimizer.step()
    if verbose:
        print(f"  pretrained for {n_pretrain_epochs} epochs (reconstruction-only)")
    check_health(model, X_train, device, "after pretraining")

    center = compute_svdd_center(model, X_train, device=device)

    # Stage 2: joint training on L_total = L_svdd + lambda1 * L_kl + lambda2 * L_rec
    model.train()
    history = []
    for epoch in range(n_train_epochs):
        totals = {"total": 0.0, "recon": 0.0, "svdd": 0.0, "kl": 0.0}
        n_seen = 0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            x_hat, z_hat = model(batch)
            recon = F.mse_loss(x_hat, batch)
            svdd = svdd_score(z_hat, center).mean()
            kl = kl_to_standard_normal(z_hat).mean()
            loss = svdd + lambda1 * kl + lambda2 * recon
            loss.backward()
            optimizer.step()

            bs = batch.size(0)
            totals["total"] += loss.item() * bs
            totals["recon"] += recon.item() * bs
            totals["svdd"] += svdd.item() * bs
            totals["kl"] += kl.item() * bs
            n_seen += bs

        for k in totals:
            totals[k] /= n_seen
        history.append(totals)
        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f"  epoch {epoch+1:3d}/{n_train_epochs}  total={totals['total']:.4f}  "
                  f"recon={totals['recon']:.5f}  svdd={totals['svdd']:.4f}  kl={totals['kl']:.4f}")
    ratio, spread = check_health(model, X_train, device, "after joint training")
    history[-1]["recon_ratio"], history[-1]["latent_spread"] = ratio, spread

    return center, history

## 6. Scoring: anomaly score -> FOM and AUROC

Each evaluation pulse is scored with `‖ẑ − c‖²`. The score is right-skewed, so we take its log, fit a two-component Gaussian mixture, and compute FOM from the two fitted peaks the same way the dataset paper does.
**AUROC** is also reported, for the SVDD score and for the plain reconstruction error. AUROC needs no mixture fit and no threshold, and 0.5 means "no separation".

**Degenerate fits.** If the scores collapse to one value the mixture fit is meaningless and FOM can come out absurdly large (the first version printed 97.8 for a model whose scores were *identical* for every pulse).
`fit_two_gaussian_fom` flags this when the scores are nearly identical, when one mixture component holds almost none of the pulses, or when one fitted peak is razor-thin.

**FOM is a weak guide here.** It rests on a two-Gaussian fit, which can look good or bad for a score that doesn't separate the classes: in the seed table in Section 8 the seed with the *lowest* AUROC has the *highest* FOM. Judge by AUROC and the agreement line first.

In [ ]:
def fit_two_gaussian_fom(score, y=None, random_state=0, reg_covar=1e-4):
    # log-transform a positive anomaly score, fit a 2-component GMM, compute FOM = S / (FWHM_0 + FWHM_1).
    log_score = np.log(score + 1e-8)
    gm = GaussianMixture(n_components=2, random_state=random_state, n_init=10, reg_covar=reg_covar)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")              # sklearn warns loudly on collapsed data; we flag it ourselves below
        gm.fit(log_score.reshape(-1, 1))
    means_raw = gm.means_.flatten()
    sigmas_raw = np.sqrt(gm.covariances_.flatten())
    order = np.argsort(means_raw)
    means, sigmas = means_raw[order], sigmas_raw[order]
    S = means[1] - means[0]
    fwhm = sigmas * 2.35482
    FOM = S / (fwhm[0] + fwhm[1])

    data_range = log_score.max() - log_score.min()
    reasons = []
    if data_range < 1e-3:
        reasons.append("the scores are (nearly) identical for every pulse")
    if gm.weights_.min() < 0.02:
        reasons.append(f"one mixture component holds only {100 * gm.weights_.min():.1f}% of the pulses")
    if fwhm.min() < 0.01 * max(data_range, 1e-8):
        reasons.append(f"the tighter fitted peak (sigma={sigmas.min():.1e}) is far narrower than the data's spread ({data_range:.2f})")
    degenerate = bool(reasons)

    out = {"FOM": FOM, "log_score": log_score, "means": means, "sigmas": sigmas,
           "fwhm": fwhm, "degenerate": degenerate, "weights": gm.weights_[order]}
    if y is not None:
        neutron_cluster = np.argmax(means_raw)
        pred_is_neutron = (gm.predict(log_score.reshape(-1, 1)) == neutron_cluster).astype(int)
        out["agreement"] = (pred_is_neutron == y).mean()

    if degenerate:
        print(f"  [!] FOM={FOM:.1f} is unreliable: " + "; ".join(reasons) + ".")
    return out


def evaluate_iae(model, center, X, y=None, mask=None, device="cpu"):
    # Score pulses with SCORE = ||z_hat - c||^2, then compute FOM (+ AUROC of SVDD score and reconstruction error if y is given).
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1).to(device)
        x_hat, z_hat = model(X_t)
        score = svdd_score(z_hat, center).cpu().numpy()
        rec_err = ((x_hat - X_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
    if mask is not None:
        score, rec_err = score[mask], rec_err[mask]
        y = None if y is None else y[mask]
    out = fit_two_gaussian_fom(score, y=y)
    out["rec_err"] = rec_err
    if y is not None:
        out["auroc_svdd"] = roc_auc_score(y, score)
        out["auroc_recon"] = roc_auc_score(y, rec_err)
        out["y"] = y
    return out

## 7. Train and evaluate

Two switches let you reproduce the original failure for yourself: set `NORMALISE_PULSES = False` and `STABILISE = False` (and `LATENT_DIM = 2`) and the health check and the degenerate flag will both fire.
`λ1 = 0.01`, `λ2 = 1.0`, 15 + 15 epochs and `LATENT_DIM = 8` are starting points chosen from a small, seed-limited set of runs -- not carefully tuned values.

In [ ]:
NORMALISE_PULSES = True    # False -> raw baseline-corrected pulses (the original, failing setup)
STABILISE = True           # False -> plain conv net without BatchNorm (the original, failing setup)
LATENT_DIM = 8
LAMBDA1, LAMBDA2 = 0.01, 1.0
N_PRETRAIN, N_TRAIN = 15, 15
SEED = 0

X_model = Xn if NORMALISE_PULSES else Xa

def run_once(seed, verbose=True):
    torch.manual_seed(seed)
    model = IAE_LSTM_KL(N_SAMPLES, latent_dim=LATENT_DIM, stabilise=STABILISE).to(device)
    center, history = train_iae_lstm_kl(
        model, X_model[train_mask],
        n_pretrain_epochs=N_PRETRAIN, n_train_epochs=N_TRAIN,
        lambda1=LAMBDA1, lambda2=LAMBDA2, device=device, verbose=verbose,
    )
    result = evaluate_iae(model, center, X_model, y=y, mask=eval_mask, device=device)
    return model, center, history, result

print(f"Training IAE-LSTM-KL (latent_dim={LATENT_DIM}, normalise={NORMALISE_PULSES}, stabilise={STABILISE}) on {int(train_mask.sum())} gamma pulses...\n")
model, center, history, result = run_once(SEED)
cc_auroc = roc_auc_score(y[eval_mask], cc_factor(Xa, peak_ref)[eval_mask])

print("\n--- held-out evaluation (unseen gammas + all neutrons) ---")
print(f"SVDD score:           AUROC = {result['auroc_svdd']:.3f}   FOM = {result['FOM']:.3f}"
      + ("   [degenerate -- see Section 6]" if result["degenerate"] else ""))
print(f"reconstruction error: AUROC = {result['auroc_recon']:.3f}")
print(f"reference, classic charge-comparison (tail/total): AUROC = {cc_auroc:.3f}")
print("paper's classical baselines -> ZC 1.01 | FEPS 0.80 | FGA 0.95 | CC 1.33 | LG 1.48 | PCNN 1.66 | HQC-SCM 1.79")
print(f"unsupervised-cluster vs true-label agreement: {result['agreement']*100:.1f}%  (50% would be chance)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(result["log_score"], bins=60, density=True, alpha=0.6, color="seagreen", label="log(SVDD distance score)")
xs = np.linspace(result["log_score"].min(), result["log_score"].max(), 300)
for m_, s_, name in zip(result["means"], result["sigmas"], ["gamma-like", "neutron-like"]):
    ax.plot(xs, norm.pdf(xs, m_, s_), lw=2, label=f"{name} fit")
ax.set_xlabel("log(||z_hat - c||^2)"); ax.set_ylabel("density")
ax.set_title(f"SVDD score (AUROC {result['auroc_svdd']:.2f}, FOM {result['FOM']:.2f})")
ax.legend()

ax = axes[1]   # diagnostic only: colours use the true labels, which the model never saw
log_rec = np.log(result["rec_err"] + 1e-12)
bins = np.linspace(log_rec.min(), log_rec.max(), 60)
ax.hist(log_rec[result["y"] == 0], bins=bins, density=True, alpha=0.6, color="tab:blue", label="gamma (held out)")
ax.hist(log_rec[result["y"] == 1], bins=bins, density=True, alpha=0.6, color="tab:red", label="neutron")
ax.set_xlabel("log(reconstruction error)"); ax.set_ylabel("density")
ax.set_title(f"Reconstruction error (AUROC {result['auroc_recon']:.2f})")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Seed spread (read before trusting any single number)

The result depends noticeably on the random seed -- in the debugging runs the SVDD-score AUROC ranged from 0.53 to 0.86 across seeds with identical settings. This cell trains a few more seeds and tabulates them.

In [ ]:
N_EXTRA_SEEDS = 2   # extra seeds to train; set to 0 to skip (each takes about as long as the main run)

rows = [(SEED, history[-1]["recon_ratio"], result)]
for s in range(SEED + 1, SEED + 1 + N_EXTRA_SEEDS):
    print(f"--- seed {s} ---")
    _, _, h, r = run_once(s, verbose=False)
    rows.append((s, h[-1]["recon_ratio"], r))

print(f"\n{'seed':>4}  {'recon/baseline':>14}  {'SVDD AUROC':>10}  {'recon AUROC':>11}  {'FOM':>6}  note")
for s, ratio, r in rows:
    print(f"{s:>4}  {ratio:>14.3f}  {r['auroc_svdd']:>10.3f}  {r['auroc_recon']:>11.3f}  {r['FOM']:>6.2f}  "
          + ("DEGENERATE FOM" if r["degenerate"] else ""))

## Notes from the debugging session

Everything below was measured on the real TAMU data (the two `validation` CSVs: 2197 gammas + 2197 neutrons) and scored on held-out gammas plus all neutrons (AUROC needs the true labels, so they are used for scoring only).
Treat the numbers as indicative: a handful of seeds, and latent size and loss weights were looked at against held-out AUROC.

**Why the first version collapsed.** Pulses were in raw ADC units (peaks 49-992). Within ~20 Adam steps the pre-LSTM latent grew from std ~0.6 to ~150 and the gate pre-activations went far into saturation; from then on the encoder received exactly zero gradient
and the decoder learned only the average pulse (loss 301.07 vs 301.069 for the average pulse). The SVDD center came out as `[0.1, 0.7616]` (0.7616 = tanh(1), the largest value the module can emit), which is why `svdd` read exactly 0.0100 and `kl` exactly 0.2900.
Every score was identical, so the mixture fit left one component empty at mean 0 -- producing the spurious FOM of 97.782 that the old flag could not see.

**Input-scale window.** The plain network only trains inside a window of input scales. Raw ADC units: gates saturate (latent std exactly 0). Peak-normalised pulses at their natural std (~0.09): gates are fine but 27/64 last-layer and 12/32 middle-layer encoder channels die and the latent is constant.
Unit-std pulses: trained on 2 of 3 seeds at `latent_dim=2` and 0 of 3 at `latent_dim=8`. With the BatchNorm stabilisers every one of the 17 runs for which I recorded pretraining health trained (recon/baseline 0.19-0.56 afterwards), and raw ADC units trained too.
The stabilisers do not make the *joint* stage safe, though: peak-normalised pulses at std 0.09 still collapsed there (the reconstruction term is tiny next to the SVDD term), so keep the pulses at unit scale.

**Amplitude.** With amplitude kept, the 2-D latent correlated ~-0.99 with peak height and ~0 with tail/total, while amplitude is not a class confound (AUROC 0.46); the SVDD distance then sat at chance (AUROC 0.47-0.49).

**What this version achieves** (held-out, `latent_dim=8`, seeds 0-3): SVDD-score AUROC 0.86 / 0.67 / 0.53 / 0.67 (mean 0.68); reconstruction-error AUROC 0.90 / 0.93 / 0.83 / 0.94 (mean 0.90). The classic tail/total charge ratio gets 0.99 on the same pulses.
So the SVDD score is seed-sensitive, and the reconstruction error is steadier but still clearly below a plain tail/total ratio. The joint stage can also pull the reconstruction away from its pretrained quality
(recon/baseline went from ~0.25 after pretraining to 0.29 / 0.46 / 0.62 / 0.26 for seeds 0-3).

**Latent size.** `latent_dim=2` trains but its SVDD score stayed at chance in three seeds (0.50-0.55, earlier runs). Sizes 8, 16 and 32 could not be separated given the seed-to-seed spread
(SVDD AUROC means 0.68 / 0.73 / 0.77 over four seeds each; ranges 0.53-0.86 / 0.63-0.88 / 0.56-0.90). The worst final reconstruction was 0.62x the average-pulse baseline at 8, 0.79x at 32 and 2.2x at 16, which is why 8 is the default. `latent_dim=128` was worse (0.42-0.55, earlier runs).

**KL.** In the 2-D-latent, unstabilised runs the latent spread after joint training was 0.035 / 0.035 / 0.030 at `λ1` = 0 / 0.01 / 1, so the term did not stop the ball shrinking, and held-out SVDD AUROC did not improve.
A Gaussian KL computed from the batch mean and variance does hold the spread open (0.63 vs 0.04 at `λ1`=0.1, one seed) but lowered the SVDD AUROC (0.56 vs 0.75). `λ1` is left at 0.01 and treated as inert.

**Caveats.** Model choices were informed by held-out pulses, so quote a final number only from a separate validation slice. On a GPU, convolutions aren't perfectly deterministic even with seeds set, so re-running can shift results slightly.